# T4SA Preprocessing with Negation Handling
This notebook processes the T4SA dataset with negation handling for better sentiment detection.

In [1]:
# Step 1: Import libraries
import re
import numpy as np
import pandas as pd
import pickle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
# Step 2: Load datasets
scores_df = pd.read_csv("./datasetFolder/t4sa_text_sentiment.tsv", sep="\t", names=["TWID", "NEG", "NEU", "POS"], skiprows=1)
tweets_df = pd.read_csv("./datasetFolder/raw_tweets_text.csv")
merged_df = pd.merge(tweets_df, scores_df, left_on="id", right_on="TWID")

def get_label(row):
    scores = {"NEG": row["NEG"], "NEU": row["NEU"], "POS": row["POS"]}
    return max(scores, key=scores.get)

label_map = {"NEG": 0, "POS": 1, "NEU": 2}
merged_df["label"] = merged_df.apply(get_label, axis=1).map(label_map)
final_df = merged_df[["text", "label"]]

In [3]:
# Step 3: Clean and handle negations
NEGATION_WORDS = {"not", "no", "never", "n't"}

def clean_and_negate(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r"@\w+|#\w+", '', text)
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()
    new_words = []
    negate = False
    for word in words:
        if word in NEGATION_WORDS:
            negate = True
            new_words.append(word)
            continue
        if negate:
            new_words.append("not_" + word)
            negate = False
        else:
            new_words.append(word)
    return ' '.join(new_words)

final_df["clean_text"] = final_df["text"].astype(str).apply(clean_and_negate)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13952\3188809128.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df["clean_text"] = final_df["text"].astype(str).apply(clean_and_negate)


In [4]:
# Step 4: Tokenize and save
texts = final_df["clean_text"].tolist()
labels = final_df["label"].values

vocab_size = 20000
max_len = 50

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

np.save("savedPackages/padded_sequences_withNegations.npy", padded_sequences)
np.save("savedPackages/labels_withNegations.npy", labels)

with open("savedPackages/tokenizer_withNegations.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("✅ Saved padded_sequences.npy, labels.npy, and tokenizer.pkl")

✅ Saved padded_sequences.npy, labels.npy, and tokenizer.pkl
